# Сборка датасета для модели сутяжности клиента

Пайплайн вынесен в пакет `querulus.dataset`. Тетрадка только настраивает окружение и вызывает `run_pipeline`.

## Настройка

In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
from pathlib import Path

# Корень репозитория и src/ — как в integration/tests/__init__.py
_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
import logging

import pandas as pd

from querulus.dataset import run_pipeline
from querulus.dataset.io import setup_notebook_logging

setup_notebook_logging(logging.INFO)

# True: подгрузить готовый финальный датасет; False: пересобрать пайплайн
LOAD_FINAL_DATASET = True
FINAL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"

# False: parquet из data/raw (или SQL при отсутствии); True: всегда SQL
USE_SQL = False
SAVE_CHECKPOINT = True

## Пайплайн

In [ ]:
if LOAD_FINAL_DATASET:
    if not FINAL_DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Финальный датасет не найден: {FINAL_DATASET_PATH}. "
            "Поставьте LOAD_FINAL_DATASET=False, чтобы собрать его заново."
        )
    df = pd.read_parquet(FINAL_DATASET_PATH)
    print(f"LOAD final dataset: {FINAL_DATASET_PATH} shape={df.shape}")
else:
    df = run_pipeline(use_sql=USE_SQL, save_checkpoint=SAVE_CHECKPOINT)

## Проверка результата

In [ ]:
df.shape

In [ ]:
df[["TARGET", "TARGET_2"]].value_counts(dropna=False)

## Обучение моделей

Обучение вынесено в отдельный пакет `querulus.training`. Сборка датасета и обучение запускаются разными ячейками.

In [ ]:
import importlib

importlib.invalidate_caches()

from IPython.display import Markdown, display

from querulus.training.config import TrainingConfig
from querulus.training.pipeline import format_metrics_table, train_models
from querulus.training.selected_features import (
    DEFAULT_FREQUENCY_FEATURES,
    DEFAULT_SEVERITY_FEATURES,
)

# None = все MVP-признаки; по умолчанию — финальные списки из config_cf_3 / config_rg_3.
FREQUENCY_FEATURES = DEFAULT_FREQUENCY_FEATURES
SEVERITY_FEATURES = DEFAULT_SEVERITY_FEATURES

TRAINING_CONFIG = TrainingConfig(
    frequency_features=FREQUENCY_FEATURES,
    severity_features=SEVERITY_FEATURES,
)
training = train_models(df, TRAINING_CONFIG)

display(Markdown("### Frequency (classification)"))
display(format_metrics_table(training.frequency_metrics_table))
display(Markdown("### Severity (regression)"))
display(format_metrics_table(training.severity_metrics_table))

In [ ]:
display(training.frequency_importance.head(30))
display(training.severity_importance.head(30))